In [2]:
import fastf1 as ff
import numpy as np

# Stocker le cache
ff.Cache.enable_cache("donneescachef1")

# Charger la course spécifique (Bahrein, 2024) + données pilote
session = ff.get_session(2024, "Bahrein", "R")  
session.load()
laps_ver = session.laps.pick_driver("VER")
clean_laps = laps_ver.pick_quicklaps().pick_track_status("1") # On garde uniquement les tours sous drapeau vert 
chronos_secondes = clean_laps["LapTime"].dt.total_seconds().dropna() # Convertir en secondes pour une lecture + simple

# Calcul des paramètres de la loi Normale
mu_pilote = np.mean(chronos_secondes) # Moyenne des temps du pilote
sigma_pilote = np.std(chronos_secondes) # L'ecart type

print(f"Pilote : Max VERSTAPPEN (Bahrein, 2024)")
print(f"Nb de tours clean : {len(chronos_secondes)}")
print(f"Temps de base mu : {round(mu_pilote, 3)} sec")
print(f"Régularité sigma : {round(sigma_pilote, 3)} sec")


events      WARNING 	Correcting user input 'Bahrein' to 'Bahrain Grand Prix'
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
re

Pilote : Max VERSTAPPEN (Bahrein, 2024)
Nb de tours clean : 50
Temps de base mu : 95.6 sec
Régularité sigma : 1.04 sec


/Users/maximilien/PERSO 🙈/PROJET-QUANT-FINANCE/PROJET-GITHUB/.venv/lib/python3.12/site-packages/fastf1/core.py:3175: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"


In [3]:
import pandas as pd 

# Préparation des variables t_base, Kfuel(N-L) et a(L) pour la régression

df_reg = pd.DataFrame() # On crée un tableau vide pour la régression

df_reg["LapTime_Y"] = clean_laps["LapTime"].dt.total_seconds() # Variable pour le chronos en secondes

df_reg["L"] = clean_laps["LapNumber"] # Variable de repère temporel L, le numéro de tours

N = 57 # Distance total de Bahrein (N) = 57 tours
df_reg["Fuel_Remaining_X1"] = N - df_reg['L'] # Variable pour le carburant restant (en tours)

df_reg["Tyre_Age_X2"] = clean_laps["TyreLife"] # Variable pour l'âge du pneu, récup dans fastf1 (en tours)

df_reg = df_reg.dropna() # removing missing value 

print(df_reg.head())

   LapTime_Y    L  Fuel_Remaining_X1  Tyre_Age_X2
1     96.296  2.0               55.0          5.0
2     96.753  3.0               54.0          6.0
3     96.647  4.0               53.0          7.0
4     97.173  5.0               52.0          8.0
5     97.092  6.0               51.0          9.0


In [4]:
import statsmodels.api as sm 

# On sépare les variables (X) explicatives de la cible (Y)
X = df_reg[["Fuel_Remaining_X1", "Tyre_Age_X2"]]
Y = df_reg["LapTime_Y"]

X = sm.add_constant(X) # On ajoute notre constante, t_base

# Création et résolution du modèle OLS
modele = sm.OLS(Y, X)
resultats = modele.fit()

print(resultats.summary()) # Affichage du rapport statistique complet 

# Exract propre pour le projet 
t_base = resultats.params["const"]
k_fuel = resultats.params["Fuel_Remaining_X1"]
rho = resultats.params["Tyre_Age_X2"]


print(f"t_base (Rythme absolu sans contrainte) : {round(t_base, 3)}s")
print(f"k_fuel (gain lié à l'essence/tour) : {round(k_fuel, 3)}s")
print(f"rho (Perte lié à la gomme par tour) : {round(rho, 3)}s")

                            OLS Regression Results                            
Dep. Variable:              LapTime_Y   R-squared:                       0.778
Model:                            OLS   Adj. R-squared:                  0.768
Method:                 Least Squares   F-statistic:                     82.25
Date:                Thu, 23 Jul 2026   Prob (F-statistic):           4.46e-16
Time:                        17:30:51   Log-Likelihood:                -35.327
No. Observations:                  50   AIC:                             76.65
Df Residuals:                      47   BIC:                             82.39
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                93.0244      0.23